In [1]:
from transformers import AutoTokenizer

/home/jovyan/Works/MSc_Thesis/mh_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_name = "meta-llama/Llama-3.2-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [3]:
def prepare_dataset(dataset):
   
    
    # Data prompt
    data_prompt = """You are a helpful mental health assistant. When users ask for health data, use the available functions to retrieve the information. Think through your response before choosing a function. Use <think>your reasoning</think> tags to show your thought process, then use <function>json</function> tags to call the appropriate function.

### Input:
Patient ID: {}
Pateint Query: {}

### Response:
{}"""

    # Formatting the prompt
    EOS_TOKEN = tokenizer.eos_token
    def formatting_prompt(examples):
        patient_ids = examples["patient_id"]
        queries = examples["patient_query"]
        responses = examples["assistant_response"]
        texts = []
        
        for patient_id, query, response in zip(patient_ids, queries, responses):
            text = data_prompt.format(patient_id, query, response) + EOS_TOKEN
            texts.append(text)
        
        return {"text": texts}
    
    dataset_with_text_column = dataset.map(formatting_prompt, batched=True)
    
    return dataset_with_text_column



In [30]:
# Load the dataset
from datasets import load_dataset, DatasetDict, load_from_disk
dataset_path = "/home/jovyan/Works/MSc_Thesis/dataset/mh_function_calling_dataset_v2.csv"
data_file = {"train":dataset_path}
fc_dataset = load_dataset("csv",data_files=data_file)
fc_dataset

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 5000 examples [00:00, 90959.45 examples/s]


DatasetDict({
    train: Dataset({
        features: ['patient_query', 'assistant_response', 'function_name', 'patient_id', 'num_days', 'has_temporal', 'condition'],
        num_rows: 5000
    })
})

In [23]:
formatted_fc_dataset = prepare_dataset(fc_dataset)
formatted_fc_dataset

DatasetDict({
    train: Dataset({
        features: ['patient_query', 'assistant_response', 'function_name', 'patient_id', 'num_days', 'has_temporal', 'condition', 'text'],
        num_rows: 5000
    })
})

In [24]:
# One sample
formatted_fc_dataset["train"][0]

{'patient_query': "Been dealing with restlessness in recent weeks - what's my exercise heart rate like?",
 'assistant_response': '<think>\nSince they said \'in recent weeks\' and need exercise heart rate data, I should use get_exercise_heart_rate_data to get those measurements. I\'ll get about 7 days of exercise heart rate data to show recent patterns related to their restlessness.\n</think>\n\n<function>\n{\n  "name": "get_exercise_heart_rate_data",\n  "parameters": {\n    "patient_id": "P5713",\n    "num_days": 7\n  }\n}\n</function>',
 'function_name': 'get_exercise_heart_rate_data',
 'patient_id': 'P5713',
 'num_days': 7,
 'has_temporal': False,
 'condition': 'restlessness',
 'text': 'You are a helpful mental health assistant. When users ask for health data, use the available functions to retrieve the information. Think through your response before choosing a function. Use <think>your reasoning</think> tags to show your thought process, then use <function>json</function> tags to ca

In [25]:
# One sample text field
print(formatted_fc_dataset["train"][0]["text"])

You are a helpful mental health assistant. When users ask for health data, use the available functions to retrieve the information. Think through your response before choosing a function. Use <think>your reasoning</think> tags to show your thought process, then use <function>json</function> tags to call the appropriate function.

### Input:
Patient ID: P5713
Pateint Query: Been dealing with restlessness in recent weeks - what's my exercise heart rate like?

### Response:
<think>
Since they said 'in recent weeks' and need exercise heart rate data, I should use get_exercise_heart_rate_data to get those measurements. I'll get about 7 days of exercise heart rate data to show recent patterns related to their restlessness.
</think>

<function>
{
  "name": "get_exercise_heart_rate_data",
  "parameters": {
    "patient_id": "P5713",
    "num_days": 7
  }
}
</function><|eot_id|>


In [28]:
# Split the dataset into train and test (80% train, 20% test)
train_temp_split = formatted_fc_dataset["train"].train_test_split(train_size=0.8, seed=42)
val_test_split = train_temp_split["test"].train_test_split(train_size=0.5, seed=42)

split_formatted_fc_dataset = {
    "train": train_temp_split["train"],
    "validation": val_test_split["train"],  # First half of the temp split
    "test": val_test_split["test"]          # Second half of the temp split
}

split_formatted_fc_dataset = DatasetDict(split_formatted_fc_dataset)

In [27]:
# Save to disk
dataset_based_path = "/home/jovyan/Works/MSc_Thesis/dataset"
split_formatted_fc_dataset.save_to_disk(f"{dataset_based_path}/mh-function-calling-dataset-hf-v2")

Saving the dataset (1/1 shards): 100%|██████████| 500/500 [00:00<00:00, 37857.46 examples/s]


In [31]:
dataset = load_from_disk(f"{dataset_based_path}/mh-function-calling-dataset-hf-v2")

In [37]:
dataset["test"][111]

{'patient_query': "Been dealing with sadness in the recent past - what's my power like?",
 'assistant_response': '<think>\nSince they said \'in the recent past\' and need power data, I should use get_power_data to get those measurements. I\'ll get about 7 days of power data to show recent patterns related to their sadness.\n</think>\n\n<function>\n{\n  "name": "get_power_data",\n  "parameters": {\n    "patient_id": "UID_9947",\n    "num_days": 7\n  }\n}\n</function>',
 'function_name': 'get_power_data',
 'patient_id': 'UID_9947',
 'num_days': 7,
 'has_temporal': False,
 'condition': 'sadness',
 'text': 'You are a helpful mental health assistant. When users ask for health data, use the available functions to retrieve the information. Think through your response before choosing a function. Use <think>your reasoning</think> tags to show your thought process, then use <function>json</function> tags to call the appropriate function.\n\n### Input:\nPatient ID: UID_9947\nPateint Query: Been d